In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


In [2]:
import numpy as np
from utils.LDPC_encode import QCLDPCEncoder

base_pc_matrix = '../pc_matrices/matlab_h.txt'
P = np.loadtxt(base_pc_matrix, dtype=int)
blocksize = 27

encoder = QCLDPCEncoder(base_matrix= P, Z= blocksize)

Initializing Encoder: Full Matrix Size 162x648, Message Bits: 486
  > Inverting Parity Matrix (this may take a moment for large Z)...
  > Computing Generator Matrix...
Encoder Ready.


In [3]:
k = (P.shape[1] - P.shape[0]) * blocksize
n_frames = 100

message = np.random.randint(0, 2, (n_frames, k))

codeword = encoder.encode(message)
tx_codeword = 1 - 2 * codeword

In [4]:
from ldpc.rbl_bp_decoder import RBLBPDecoder

decoder = RBLBPDecoder(encoder.H, max_iter=20, alpha=0.5)

In [5]:
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

snrs = [3, 4, 5, 6, 7]
bers = []

for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db= snr)
    decoded_codewords = []

    for i in range(n_frames):
        decoded_codeword = decoder.decode(rx_llrs[i, :])
        decoded_codewords.append(decoded_codeword)
    
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :k]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

KeyboardInterrupt: 